# Notebook 3A: KE and EKE Trends

This notebook focuses on **Kinetic Energy (KE)** and **Eddy Kinetic Energy (EKE)** diagnostics and trends.

It generates:
1. **2D mean KE and EKE maps** and **2D trend maps**.
2. **Regional time series with trend overlays**.
3. **Regional bar-trend summaries**.

## Trend methodology
- **Trend estimator:** Theil-Sen robust median slope.
- **Significance test:** Mann-Kendall with **autocorrelation correction** following **Yue and Wang (2004)**.
- **Significance labels:** `* p<0.05`, `** p<0.01`, `*** p<0.001`.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats

# Paths
REPO_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
if not (REPO_ROOT / 'outputs' / 'trends').exists() and (Path.cwd() / 'outputs' / 'trends').exists():
    REPO_ROOT = Path.cwd()

INPUT_TREND_DIR = REPO_ROOT / 'outputs' / 'trends'
INPUT_HP_DIR = REPO_ROOT / 'outputs' / 'monthly_half-power_points'
INPUT_2D_TRENDS_FILE = INPUT_TREND_DIR / 'ke_eke_2d_trends.nc'
PLOT_DIR = REPO_ROOT / 'outputs' / 'plots' / 'trends_split'
FIG_DIR = REPO_ROOT / 'figures' / 'manuscript'   # publication figures (tracked by git)
FIG_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

REGION_ORDER = ['Indian', 'Pacific', 'Atlantic', 'ACC']
REGION_COLORS = {
    'Indian': '#2A9D8F',
    'Pacific': '#457B9D',
    'Atlantic': '#E07A5F',
    'ACC': '#2F3E46',
}

STYLE = {
    'raw_alpha': 0.18,
    'raw_lw': 0.7,
    'smooth_lw': 1.9,
    'trend_lw': 1.4,
    'grid_alpha': 0.18,
    'grid_lw': 0.5,
    'axis_label_size': 11,
    'title_size': 11,
    'tick_size': 9,
    'annot_size': 9,
    'panel_size': 11,
    'neutral': '#7A8288',
    'ke': "#e96c6a",
    'eke': '#2A9D8F',
    'trend_sig': '#1F2933',
}

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':         10,
    'axes.labelsize':    STYLE['axis_label_size'],
    'axes.titlesize':    STYLE['title_size'],
    'axes.titleweight':  'bold',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.facecolor':    'white',
    'axes.grid':         True,
    'axes.linewidth':    0.6,
    'figure.facecolor':  'white',
    'grid.alpha':        0.18,
    'grid.linewidth':    0.5,
    'xtick.labelsize':   STYLE['tick_size'],
    'ytick.labelsize':   STYLE['tick_size'],
    'legend.fontsize':   8,
    'legend.framealpha': 0.9,
    'legend.edgecolor':  '#cccccc',
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})


def panel_label(ax, label):
    ax.text(
        0.015,
        0.985,
        f'({label})',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=STYLE['panel_size'],
        fontweight='semibold',
        color='#1F2933',
    )


def base_timeseries_style(ax):
    ax.grid(alpha=STYLE['grid_alpha'], lw=STYLE['grid_lw'])
    ax.tick_params(length=3.0, width=0.7, color='#67727A')


def assign_sector(lon):
    lon360 = lon % 360
    if lon360 >= 290 or lon360 < 20:
        return 'Atlantic'
    if 20 <= lon360 < 147:
        return 'Indian'
    if 147 <= lon360 < 290:
        return 'Pacific'
    return 'Unknown'


def stars(p):
    if not np.isfinite(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def theil_sen_with_ci(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return {'slope': np.nan, 'intercept': np.nan, 'lo': np.nan, 'hi': np.nan, 'se': np.nan}

    res = stats.theilslopes(y[m], x[m], alpha=0.95)
    slope = float(res.slope)
    intercept = float(res.intercept)
    lo = float(res.low_slope)
    hi = float(res.high_slope)
    se = (hi - lo) / (2 * 1.96) if np.isfinite(lo) and np.isfinite(hi) else np.nan
    return {'slope': slope, 'intercept': intercept, 'lo': lo, 'hi': hi, 'se': se}


def _mk_s_var(y):
    n = len(y)
    s = 0
    for i in range(n - 1):
        s += np.sign(y[i + 1:] - y[i]).sum()

    _, counts = np.unique(y, return_counts=True)
    tie_term = np.sum(counts * (counts - 1) * (2 * counts + 5))
    var_s = (n * (n - 1) * (2 * n + 5) - tie_term) / 18.0
    return s, var_s


def mann_kendall_yue_wang(y, alpha=0.05, max_lag=12):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    n = len(y)
    if n < 5:
        return {'tau': np.nan, 'p': np.nan, 'n_eff': np.nan}

    s, var_s = _mk_s_var(y)
    tau = s / (0.5 * n * (n - 1))

    # Detrend before autocorrelation estimation to avoid trend leakage into rho.
    x = np.arange(n, dtype=float)
    ts = theil_sen_with_ci(x, y)
    if np.isfinite(ts['slope']) and np.isfinite(ts['intercept']):
        y_for_ac = y - (ts['slope'] * x + ts['intercept'])
    else:
        y_for_ac = y - np.nanmean(y)

    ranks = stats.rankdata(y_for_ac)
    r = ranks - np.mean(ranks)
    denom = np.sum(r**2)
    if denom <= 0:
        return {'tau': tau, 'p': np.nan, 'n_eff': np.nan}

    max_lag = min(max_lag, n - 2)
    sig_rhos = []
    for k in range(1, max_lag + 1):
        num = np.sum(r[:-k] * r[k:])
        rho_k = num / denom
        crit = stats.norm.ppf(1 - alpha / 2) / np.sqrt(n)
        if np.abs(rho_k) > crit:
            sig_rhos.append((k, rho_k))

    if len(sig_rhos) == 0:
        n_eff = n
    else:
        corr_sum = np.sum([(1 - k / n) * rho for k, rho in sig_rhos])
        n_eff = n / (1 + 2 * corr_sum)
        n_eff = np.clip(n_eff, 2, n)

    var_s_adj = var_s * (n / n_eff)
    if s > 0:
        z = (s - 1) / np.sqrt(var_s_adj)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s_adj)
    else:
        z = 0.0
    p = 2 * stats.norm.sf(abs(z))

    return {'tau': float(tau), 'p': float(p), 'n_eff': float(n_eff)}


print(f'Trend input: {INPUT_TREND_DIR}')
print(f'2D fields input: {INPUT_HP_DIR}')
print(f'Precomputed 2D trend file: {INPUT_2D_TRENDS_FILE}')
print(f'Output plots: {PLOT_DIR}')


Trend input: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/trends
2D fields input: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/monthly_half-power_points
Precomputed 2D trend file: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/trends/ke_eke_2d_trends.nc
Output plots: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/plots/trends_split


In [2]:
# 1) Regional time-series source (already area-weighted by region)
energy_ts = pd.read_csv(INPUT_TREND_DIR / 'energy_field_timeseries.csv')
if 'sector' in energy_ts.columns and 'region' not in energy_ts.columns:
    energy_ts = energy_ts.rename(columns={'sector': 'region'})
if 'decimal_year' not in energy_ts.columns:
    energy_ts['decimal_year'] = energy_ts['year'] + (energy_ts['month'] - 0.5) / 12.0

energy_ts['region'] = energy_ts['region'].replace({'Whole SO': 'ACC'})
energy_ts = energy_ts[energy_ts['region'].isin(REGION_ORDER)].copy()

# Rebuild ACC from contour-bounded area (between ACC ADT south/north contours).
SSH_ACC_SOUTH = -0.6
SSH_ACC_NORTH = 0.2
acc_rows = []
missing_months = []

sample_file = sorted(INPUT_HP_DIR.glob('*/*.nc'))
if len(sample_file) > 0:
    with xr.open_dataset(sample_file[0]) as ds_sample:
        hp_lat = ds_sample['latitude'].values.astype(float)
        hp_lon = ds_sample['longitude'].values.astype(float)

    with xr.open_dataset(REPO_ROOT / 'outputs' / 'mean_adt.nc') as ds_adt:
        adt = ds_adt['mean_adt'].squeeze()
        adt_lon = adt['longitude'].values.astype(float)

        # Align ADT longitude convention to monthly KE/EKE files.
        if np.nanmax(adt_lon) > 180 and np.nanmax(hp_lon) <= 180:
            adt = adt.assign_coords(longitude=((adt['longitude'] + 180.0) % 360.0) - 180.0).sortby('longitude')
        elif np.nanmax(adt_lon) <= 180 and np.nanmax(hp_lon) > 180:
            adt = adt.assign_coords(longitude=(adt['longitude'] % 360.0)).sortby('longitude')

        adt_on_hp = adt.interp(latitude=hp_lat, longitude=hp_lon, method='linear')
        acc_mask = (
            np.isfinite(adt_on_hp.values)
            & (adt_on_hp.values >= SSH_ACC_SOUTH)
            & (adt_on_hp.values <= SSH_ACC_NORTH)
        )

    if np.isfinite(acc_mask).any() and np.nansum(acc_mask) > 0:
        lat_weights = np.cos(np.deg2rad(hp_lat))[:, None]
        w2d = np.broadcast_to(lat_weights, acc_mask.shape)

        acc_months = (
            energy_ts[['year', 'month', 'decimal_year']]
            .drop_duplicates()
            .sort_values(['year', 'month'])
            .reset_index(drop=True)
        )

        for _, row in acc_months.iterrows():
            yy = int(row['year'])
            mm = int(row['month'])
            f_month = INPUT_HP_DIR / f'{yy:04d}' / f'{mm:02d}.nc'
            if not f_month.exists():
                missing_months.append((yy, mm))
                continue

            with xr.open_dataset(f_month) as ds_m:
                ke_m = ds_m['ke'].values.astype(float)
                eke_m = ds_m['eke'].values.astype(float)

            v_ke = np.isfinite(ke_m) & acc_mask
            v_eke = np.isfinite(eke_m) & acc_mask

            if np.nansum(v_ke) == 0 or np.nansum(v_eke) == 0:
                continue

            w_ke = np.where(v_ke, w2d, 0.0)
            w_eke = np.where(v_eke, w2d, 0.0)
            den_ke = np.nansum(w_ke)
            den_eke = np.nansum(w_eke)
            if den_ke <= 0 or den_eke <= 0:
                continue

            acc_rows.append({
                'year': yy,
                'month': mm,
                'decimal_year': float(row['decimal_year']),
                'mean_ke': float(np.nansum(ke_m * w_ke) / den_ke),
                'mean_eke': float(np.nansum(eke_m * w_eke) / den_eke),
                'region': 'ACC',
            })

        if len(acc_rows) > 0:
            acc_df = pd.DataFrame(acc_rows)
            energy_ts = energy_ts[energy_ts['region'] != 'ACC'].copy()
            energy_ts = pd.concat([energy_ts, acc_df], ignore_index=True)
            energy_ts = energy_ts.sort_values(['region', 'year', 'month']).reset_index(drop=True)
            print(f"ACC rebuilt from ADT contour mask: {len(acc_df)} monthly records")
            if len(missing_months) > 0:
                print(f"WARNING: missing monthly files for {len(missing_months)} ACC records")
        else:
            print('WARNING: could not rebuild ACC from contour mask; using CSV ACC values.')
    else:
        print('WARNING: ACC contour mask is empty on KE/EKE grid; using CSV ACC values.')
else:
    print('WARNING: no monthly KE/EKE files found; using CSV ACC values.')

# 2) Precomputed 2D KE/EKE means and trend products
if not INPUT_2D_TRENDS_FILE.exists():
    raise FileNotFoundError(
        f'Precomputed 2D trend file not found: {INPUT_2D_TRENDS_FILE}. '
        'Run scripts/ke_eke_2d_trends.py first.'
    )

with xr.open_dataset(INPUT_2D_TRENDS_FILE) as ds_tr:
    print(f"2D trend file dims: {dict(ds_tr.sizes)}")
    print(
        f"Trend span (decimal year): "
        f"{ds_tr.attrs.get('time_start_decimal_year', np.nan):.2f} to "
        f"{ds_tr.attrs.get('time_end_decimal_year', np.nan):.2f}"
    )

print('Regions available:', sorted(energy_ts['region'].unique()))
print(f'Regional records: {len(energy_ts)}')

ACC rebuilt from ADT contour mask: 300 monthly records
2D trend file dims: {'latitude': 240, 'longitude': 2880}
Trend span (decimal year): 2001.04 to 2024.96
Regions available: ['ACC', 'Atlantic', 'Indian', 'Pacific']
Regional records: 1200


## Figure A1: 2D KE/EKE Maps (Notebook 3 style) + ACC Time Series

This figure mirrors the 2D map layout used in Notebook 3:
- South Polar stereographic maps of mean KE and mean EKE (cm^2 s^-2) with ACC ADT boundary contours.
- Pixel-wise **Theil-Sen trends** for KE and EKE (cm^2 s^-2 yr^-1) with shared colorbars and ACC ADT boundary contours.
- A bottom panel with **ACC KE and EKE time series on the same subplot** and trend overlays (Theil-Sen + Mann-Kendall Yue-Wang).

Trend fields are loaded from a precomputed file generated outside this notebook:
- `python scripts/ke_eke_2d_trends.py`
- output: `outputs/trends/ke_eke_2d_trends.nc`

In [3]:
# Load precomputed 2D KE/EKE means and trend fields (computed by scripts/ke_eke_2d_trends.py)
with xr.open_dataset(INPUT_2D_TRENDS_FILE) as ds_tr:
    lat_1d = ds_tr['latitude'].values.astype(float)
    lon_1d = ds_tr['longitude'].values.astype(float)
    ke_mean = ds_tr['ke_mean_cm2_s2'].values.astype(float)
    eke_mean = ds_tr['eke_mean_cm2_s2'].values.astype(float)
    ke_slope = ds_tr['ke_slope_cm2_s2_yr'].values.astype(float)
    eke_slope = ds_tr['eke_slope_cm2_s2_yr'].values.astype(float)
    ke_p = ds_tr['ke_pvalue'].values.astype(float)
    eke_p = ds_tr['eke_pvalue'].values.astype(float)

# Load mean ADT field for ACC contour overlay on maps
# ACC boundaries follow Sokolov and Rintoul (2009).
SSH_ACC_SOUTH = -0.6
SSH_ACC_NORTH = 0.2
adt_ds = xr.open_dataset(REPO_ROOT / 'outputs' / 'mean_adt.nc')
adt = adt_ds['mean_adt'].squeeze()

lon_adt = adt['longitude'].values.astype(float)
lat_adt = adt['latitude'].values.astype(float)

# Convert ADT longitudes to [-180, 180) for the polar map
if np.nanmax(lon_adt) > 180:
    lon_wrapped = ((lon_adt + 180.0) % 360.0) - 180.0
    ord_lon = np.argsort(lon_wrapped)
    lon_adt = lon_wrapped[ord_lon]
    adt = adt.isel(longitude=ord_lon)

adt_vals = adt.values.astype(float)
acc_levels = [SSH_ACC_SOUTH, SSH_ACC_NORTH]

LON2D, LAT2D = np.meshgrid(lon_1d, lat_1d)
proj = ccrs.SouthPolarStereo()

# Load the saved preprocessing sea-ice mask and use it directly.
# No fallback/proxy mask is used in this figure.

if adt_vals.shape == ke_mean.shape:
    ocean_mask = np.isfinite(adt_vals)
else:
    ocean_mask = np.isfinite(ke_mean) | np.isfinite(eke_mean)

PREPROC_ICE_FILE = REPO_ROOT / 'outputs' / 'sea_ice_mask.nc'
if not PREPROC_ICE_FILE.exists():
    raise FileNotFoundError(
        f'Sea-ice mask file not found: {PREPROC_ICE_FILE}. Run Notebook 00 to generate outputs/sea_ice_mask.nc.'
    )
with xr.open_dataset(PREPROC_ICE_FILE) as ds_ice:
    if 'ice_mask' not in ds_ice:
        raise KeyError(f"'ice_mask' variable not found in {PREPROC_ICE_FILE}")
    ice_da = ds_ice['ice_mask']
    if ('latitude' in ice_da.coords) and ('longitude' in ice_da.coords):
        ice_lon = ice_da['longitude'].values.astype(float)
        if np.nanmax(ice_lon) > 180 and np.nanmax(lon_1d) <= 180:
            ice_da = ice_da.assign_coords(
                longitude=((ice_da['longitude'] + 180.0) % 360.0) - 180.0
            ).sortby('longitude')
        elif np.nanmax(ice_lon) <= 180 and np.nanmax(lon_1d) > 180:
            ice_da = ice_da.assign_coords(
                longitude=(ice_da['longitude'] % 360.0)
            ).sortby('longitude')
        ice_interp = ice_da.interp(latitude=lat_1d, longitude=lon_1d, method='nearest')
        mask_pre = ice_interp.values.astype(float) > 0.5
    else:
        mask_pre = ice_da.values.astype(float) > 0.5

if mask_pre.shape != ke_mean.shape:
    raise ValueError(
        f'Sea-ice mask shape {mask_pre.shape} does not match trend grid {ke_mean.shape}.'
    )

ice_mask = mask_pre.astype(bool)
print(f'Sea-ice mask loaded from: {PREPROC_ICE_FILE}')

ice_overlay = np.where(ice_mask, 1.0, np.nan)
ice_pct = 100.0 * float(np.nansum(ice_mask)) / float(np.nansum(ocean_mask)) if np.nansum(ocean_mask) else np.nan
print(f'Ice-mask coverage on map domain: {ice_pct:.1f}%')

# All trend values are shown; significance is conveyed via stippling.
ke_mean_plot = np.where(ice_mask, np.nan, ke_mean)
eke_mean_plot = np.where(ice_mask, np.nan, eke_mean)
ke_slope_plot = np.where(ice_mask, np.nan, ke_slope)
eke_slope_plot = np.where(ice_mask, np.nan, eke_slope)

# Shared map limits from all trend values after ice masking.
vmax_mean = np.nanpercentile(np.r_[ke_mean[np.isfinite(ke_mean)], eke_mean[np.isfinite(eke_mean)]], 99)
trend_vals = np.r_[ke_slope_plot[np.isfinite(ke_slope_plot)], eke_slope_plot[np.isfinite(eke_slope_plot)]]
if trend_vals.size == 0:
    trend_vals = np.r_[ke_slope[np.isfinite(ke_slope)], eke_slope[np.isfinite(eke_slope)]]
vmax_trend = np.nanpercentile(np.abs(trend_vals), 97)

# Stipple stride: every 8 pixels ≈ 1° spacing at 0.125° resolution.
_STIPPLE_STRIDE = 8


def map_ax_style(ax):
    ax.set_extent([-180, 180, -82, -35], crs=ccrs.PlateCarree())
    theta = np.linspace(0, 2 * np.pi, 200)
    center = np.array([0.5, 0.5])
    radius = 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.add_feature(cfeature.LAND, facecolor="#000000", edgecolor='#555555', linewidth=0.2, zorder=2)
    ax.coastlines('50m', linewidth=0.45, color='#333333')
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.30,
        color='#7A8288',
        alpha=0.45,
        linestyle=':',
    )
    gl.xlocator = plt.FixedLocator(range(-180, 181, 60))
    gl.ylocator = plt.FixedLocator([-60, -40])
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 7, 'color': '#4A4A4A'}
    gl.ylabel_style = {'size': 7, 'color': '#4A4A4A'}
    ax.set_frame_on(False)

def add_acc_adt_contours(ax):
    cs = ax.contour(
        lon_adt,
        lat_adt,
        adt_vals,
        levels=acc_levels,
        colors=['#5f5f5f', '#2f2f2f'],
        linewidths=[0.6, 0.9],
        alpha=0.65,
        transform=ccrs.PlateCarree(),
        zorder=2,
    )
    return cs


def add_sector_boundaries(ax):
    """Add Pacific/Indian/Atlantic sector boundary lines on polar map."""
    # Use the same boundaries as assign_sector(): 20E, 147E, and 290E (70W).
    sector_lons_360 = [20.0, 147.0, 290.0]
    sector_lons = [lon if lon <= 180 else lon - 360 for lon in sector_lons_360]
    lats = np.linspace(-82, -35, 200)
    for lon in sector_lons:
        ax.plot(
            np.full_like(lats, lon),
            lats,
            color='#222222',
            lw=0.85,
            ls=(0, (5, 3)),
            transform=ccrs.PlateCarree(),
            zorder=5,
            alpha=0.75,
        )

def add_ice_mask(ax):
    ax.contourf(
        lon_1d,
        lat_1d,
        ice_overlay,
        levels=[0.5, 1.5],
        colors=['#8EC5FF'],
        alpha=0.35,
        transform=ccrs.PlateCarree(),
        zorder=2,
    )
    ax.contour(
        lon_1d,
        lat_1d,
        np.where(np.isfinite(ice_overlay), 1.0, np.nan),
        levels=[0.5],
        colors=['#595959'],
        linewidths=[0.35],
        alpha=0.5,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )

fig = plt.figure(figsize=(10.8, 12))
gs = fig.add_gridspec(
    3,
    3,
    height_ratios=[1, 1, 0.8],
    width_ratios=[1, 0.2, 0.8],
    hspace=0.22,
    wspace=0.22,
)

ax1 = fig.add_subplot(gs[0, 0], projection=proj)
ax2 = fig.add_subplot(gs[0, 1:], projection=proj)
ax3 = fig.add_subplot(gs[1, 0], projection=proj)
ax4 = fig.add_subplot(gs[1, 1:], projection=proj)
ax5 = fig.add_subplot(gs[2, 0:2])
ax6 = fig.add_subplot(gs[2, 2])

for ax in [ax1, ax2, ax3, ax4]:
    map_ax_style(ax)

# (a) Mean KE
im1 = ax1.pcolormesh(
    LON2D,
    LAT2D,
    ke_mean_plot,
    transform=ccrs.PlateCarree(),
    cmap='YlGnBu',
    vmin=0,
    vmax=vmax_mean,
    shading='auto',
    rasterized=True,
)
add_acc_adt_contours(ax1)
add_ice_mask(ax1)
add_sector_boundaries(ax1)
ax1.set_title('Mean KE (2001-2025)')

# (b) Mean EKE
ax2.pcolormesh(
    LON2D,
    LAT2D,
    eke_mean_plot,
    transform=ccrs.PlateCarree(),
    cmap='YlGnBu',
    vmin=0,
    vmax=vmax_mean,
    shading='auto',
    rasterized=True,
)
cs_adt = add_acc_adt_contours(ax2)
ax2.clabel(cs_adt, fmt='%.1f', fontsize=6, inline=True)
add_ice_mask(ax2)
add_sector_boundaries(ax2)
ax2.set_title('Mean EKE (2001-2025)')

cb_mean = fig.colorbar(im1, ax=[ax1, ax2], shrink=0.42, pad=0.02, orientation='vertical')
cb_mean.set_label(r'Kinetic Energy (cm$^2$ s$^{-2}$)')
cb_mean.ax.tick_params(labelsize=8)

trend_cmap = plt.get_cmap('RdBu_r').copy()
trend_cmap.set_bad('#F3F2EF')

# (c) KE trend — full field shown in colour; dots mark p < 0.05 (Yue-Wang MK)
im3 = ax3.pcolormesh(
    LON2D,
    LAT2D,
    ke_slope_plot,
    transform=ccrs.PlateCarree(),
    cmap=trend_cmap,
    vmin=-vmax_trend,
    vmax=vmax_trend,
    shading='auto',
    rasterized=True,
    zorder=0,
)
add_acc_adt_contours(ax3)
add_ice_mask(ax3)
add_sector_boundaries(ax3)
ax3.set_title('KE trend')

# (d) EKE trend — full field shown in colour; dots mark p < 0.05 (Yue-Wang MK)
ax4.pcolormesh(
    LON2D,
    LAT2D,
    eke_slope_plot,
    transform=ccrs.PlateCarree(),
    cmap=trend_cmap,
    vmin=-vmax_trend,
    vmax=vmax_trend,
    shading='auto',
    rasterized=True,
    zorder=0,
)
add_acc_adt_contours(ax4)
add_ice_mask(ax4)
add_sector_boundaries(ax4)
ax4.set_title('EKE trend')

cb_tr = fig.colorbar(im3, ax=[ax3, ax4], shrink=0.42, pad=0.02, orientation='vertical')
cb_tr.set_label(r'Trend (cm$^2$ s$^{-2}$ yr$^{-1}$)')
cb_tr.ax.tick_params(labelsize=8)

# (e) Southern Ocean KE and EKE time series + (f) regional/ACC trend bars
# Panel (e) uses the ACC series from energy_ts (rebuilt in the setup cell from the
# ADT contour mask, -0.6 m to +0.2 m), i.e. the same series as the ACC bars in (f).
so_grp = energy_ts[energy_ts['region'] == 'ACC'].sort_values('decimal_year').copy()
x = so_grp['decimal_year'].values.astype(float)
y_ke = so_grp['mean_ke'].values.astype(float) * 1e4
y_eke = so_grp['mean_eke'].values.astype(float) * 1e4

ax5.plot(x, y_ke, color=STYLE['ke'], alpha=STYLE['raw_alpha'], marker='.', ms=2, lw=STYLE['raw_lw'])
ax5.plot(x, y_eke, color=STYLE['eke'], alpha=STYLE['raw_alpha'], marker='.', ms=2, lw=STYLE['raw_lw'])

if len(so_grp) >= 12:
    rm_ke = pd.Series(y_ke).rolling(12, center=True, min_periods=6).mean()
    rm_eke = pd.Series(y_eke).rolling(12, center=True, min_periods=6).mean()
    ax5.plot(x, rm_ke, color=STYLE['ke'], lw=STYLE['smooth_lw'], label='KE')
    ax5.plot(x, rm_eke, color=STYLE['eke'], lw=STYLE['smooth_lw'], label='EKE')

res_ke = theil_sen_with_ci(x, y_ke)
mk_ke = mann_kendall_yue_wang(y_ke)
if np.isfinite(res_ke['slope']):
    xf = np.array([np.nanmin(x), np.nanmax(x)])
    ax5.plot(xf, res_ke['intercept'] + res_ke['slope'] * xf, color=STYLE['ke'], ls='--', lw=STYLE['trend_lw'])

res_eke = theil_sen_with_ci(x, y_eke)
mk_eke = mann_kendall_yue_wang(y_eke)
if np.isfinite(res_eke['slope']):
    xf = np.array([np.nanmin(x), np.nanmax(x)])
    ax5.plot(xf, res_eke['intercept'] + res_eke['slope'] * xf, color=STYLE['eke'], ls='--', lw=STYLE['trend_lw'])

ann_ke = f"KE: {res_ke['slope']:+.3f} cm² s⁻² yr⁻¹, p={mk_ke['p']:.3f} {stars(mk_ke['p'])}"
ann_eke = f"EKE: {res_eke['slope']:+.3f} cm² s⁻² yr⁻¹, p={mk_eke['p']:.3f} {stars(mk_eke['p'])}"
ax5.text(0.55, 0.16, ann_ke, transform=ax5.transAxes, va='top', ha='left', fontsize=STYLE['annot_size'], color=STYLE['ke'])
ax5.text(0.55, 0.08, ann_eke, transform=ax5.transAxes, va='top', ha='left', fontsize=STYLE['annot_size'], color=STYLE['eke'])

ax5.set_xlabel('Year')
ax5.set_ylabel(r'Kinetic Energy (cm$^2$ s$^{-2}$)')
base_timeseries_style(ax5)
ax5.legend(loc='best', ncol=2, fontsize=8, frameon=False)

# Recompute the regional + ACC trend table in the same way as Figure A3.
trend_rows = []
for region in REGION_ORDER:
    grp = energy_ts[energy_ts['region'] == region].sort_values('decimal_year')
    x_reg = grp['decimal_year'].values.astype(float)

    for var, label in [('mean_ke', 'KE'), ('mean_eke', 'EKE')]:
        y_reg = grp[var].values.astype(float) * 1e4
        ts_res = theil_sen_with_ci(x_reg, y_reg)
        mk_res = mann_kendall_yue_wang(y_reg)
        ci95 = 1.96 * ts_res['se'] if np.isfinite(ts_res['se']) else np.nan

        trend_rows.append({
            'region': region,
            'variable': label,
            'slope': ts_res['slope'],
            'ci95': ci95,
            'mk_p': mk_res['p'],
        })

trend_reg_df = pd.DataFrame(trend_rows)
ke_reg_df = trend_reg_df[trend_reg_df['variable'] == 'KE'].set_index('region').reindex(REGION_ORDER)
eke_reg_df = trend_reg_df[trend_reg_df['variable'] == 'EKE'].set_index('region').reindex(REGION_ORDER)

# Diagnostic check: bar panel uses ACC from energy_ts (ACC contour mask), not Whole SO.
acc_ke_bar = float(ke_reg_df.loc['ACC', 'slope'])
acc_eke_bar = float(eke_reg_df.loc['ACC', 'slope'])
print(f'Bar panel ACC slopes (ACC mask): KE={acc_ke_bar:+.3f}, EKE={acc_eke_bar:+.3f}')
print(f'Panel (e) ACC-mask slopes: KE={res_ke["slope"]:+.3f}, EKE={res_eke["slope"]:+.3f}')

xpos = np.arange(len(REGION_ORDER))
w = 0.34
ax6.bar(
    xpos - w / 2,
    ke_reg_df['slope'].values,
    yerr=ke_reg_df['ci95'].values,
    width=w,
    color=STYLE['ke'],
    edgecolor='white',
    linewidth=0.6,
    error_kw={'lw': 1.0, 'capsize': 2.8, 'capthick': 1.0},
    zorder=3,
    label='KE',
)
ax6.bar(
    xpos + w / 2,
    eke_reg_df['slope'].values,
    yerr=eke_reg_df['ci95'].values,
    width=w,
    color=STYLE['eke'],
    edgecolor='white',
    linewidth=0.6,
    error_kw={'lw': 1.0, 'capsize': 2.8, 'capthick': 1.0},
    zorder=3,
    label='EKE',
)

yr = np.nanmax(np.abs(np.r_[ke_reg_df['slope'].values, eke_reg_df['slope'].values]))
if not np.isfinite(yr):
    yr = 0.0
yr = max(yr, 1e-6)
for i, region in enumerate(REGION_ORDER):
    for xbar, sdf in [(xpos[i] - w / 2, ke_reg_df), (xpos[i] + w / 2, eke_reg_df)]:
        s = sdf.loc[region, 'slope']
        e = sdf.loc[region, 'ci95'] if np.isfinite(sdf.loc[region, 'ci95']) else 0.0
        p = sdf.loc[region, 'mk_p']
        st = stars(p)
        if st and np.isfinite(s):
            ytxt = s + e + 0.01 * yr if s >= 0 else s - e - 0.01 * yr
            ax6.text(
                xbar,
                ytxt,
                st,
                ha='center',
                va='bottom' if s >= 0 else 'top',
                fontsize=9.5,
                fontweight='semibold',
                color='#1F2933',
            )

ax6.axhline(0, color='#495057', lw=0.9, zorder=2)
ax6.set_xticks(xpos)
ax6.set_xticklabels(REGION_ORDER, rotation=25, ha='right')
ax6.set_ylabel(r'Trend (cm$^2$ s$^{-2}$ yr$^{-1}$)')
ax6.grid(axis='y', alpha=STYLE['grid_alpha'], lw=STYLE['grid_lw'])
ax6.legend(frameon=False, fontsize=7, loc='upper left')

fig.tight_layout(rect=[0, 0, 1, 0.97])
# Manuscript figure 'KE_EKE_trends' (main article)
out_fig = FIG_DIR / 'KE_EKE_trends.png'
fig.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_fig}')
adt_ds.close()


Sea-ice mask loaded from: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/sea_ice_mask.nc
Ice-mask coverage on map domain: 29.2%


Bar panel ACC slopes (ACC mask): KE=+1.321, EKE=+1.977
Panel (e) ACC-mask slopes: KE=+1.321, EKE=+1.977


/sessions/laughing-compassionate-darwin/tmp/ipykernel_10/3173272275.py:406: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.97])


Saved: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/figures/manuscript/KE_EKE_trends.png


## Figures A2 and A3: Regional Time Series and Bar Trends

- **Figure A2:** Regional KE/EKE time series with Theil-Sen trend overlays and Mann-Kendall (Yue & Wang, 2004) significance.
- **Figure A3:** Regional bar trends (Theil-Sen slopes) with 95% CI and significance stars from Mann-Kendall (Yue & Wang, 2004).

In [4]:
regions = REGION_ORDER.copy()

# -----------------------------
# Figure A2: time series trends
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.2), sharex=True)

for c, region in enumerate(regions[:-1]):
    grp = energy_ts[energy_ts['region'] == region].sort_values('decimal_year').copy()

    ax = axes[c]
    x = grp['decimal_year'].values.astype(float)
    y_ke = grp['mean_ke'].values.astype(float) * 1e4
    y_eke = grp['mean_eke'].values.astype(float) * 1e4

    ax.plot(x, y_ke, color=STYLE['ke'], alpha=STYLE['raw_alpha'], marker='.', ms=2.0, lw=STYLE['raw_lw'])
    ax.plot(x, y_eke, color=STYLE['eke'], alpha=STYLE['raw_alpha'], marker='.', ms=2.0, lw=STYLE['raw_lw'])

    if len(grp) >= 12:
        rm_ke = pd.Series(y_ke).rolling(12, center=True, min_periods=6).mean()
        rm_eke = pd.Series(y_eke).rolling(12, center=True, min_periods=6).mean()
        ax.plot(x, rm_ke, color=STYLE['ke'], lw=STYLE['smooth_lw'])
        ax.plot(x, rm_eke, color=STYLE['eke'], lw=STYLE['smooth_lw'])

    ts_res_ke = theil_sen_with_ci(x, y_ke)
    ts_res_eke = theil_sen_with_ci(x, y_eke)
    mk_res_ke = mann_kendall_yue_wang(y_ke)
    mk_res_eke = mann_kendall_yue_wang(y_eke)

    if np.isfinite(ts_res_ke['slope']):
        xf = np.array([np.nanmin(x), np.nanmax(x)])
        trend_col_ke = STYLE['trend_sig'] if (np.isfinite(mk_res_ke['p']) and mk_res_ke['p'] < 0.05) else STYLE['neutral']
        trend_col_eke = STYLE['trend_sig'] if (np.isfinite(mk_res_eke['p']) and mk_res_eke['p'] < 0.05) else STYLE['neutral']
        ax.plot(xf, ts_res_ke['intercept'] + ts_res_ke['slope'] * xf, color=STYLE['ke'], ls='--', lw=STYLE['trend_lw'])
        ax.plot(xf, ts_res_eke['intercept'] + ts_res_eke['slope'] * xf, color=STYLE['eke'], ls='--', lw=STYLE['trend_lw'])

        txt_ke = f"KE: {ts_res_ke['slope']:+.3f} cm² s⁻² yr⁻¹, p={mk_res_ke['p']:.3f} {stars(mk_res_ke['p'])}"
        txt_eke = f"EKE: {ts_res_eke['slope']:+.3f} cm² s⁻² yr⁻¹, p={mk_res_eke['p']:.3f} {stars(mk_res_eke['p'])}"
        ax.text(
            0.98,
            0.02,
            txt_ke,
            transform=ax.transAxes,
            ha='right',
            va='bottom',
            fontsize=STYLE['annot_size'],
            color=STYLE['ke'],
            bbox=dict(facecolor='white', alpha=0.82, edgecolor='none', boxstyle='round,pad=0.2'),
        )
        ax.text(
            0.98,
            0.08,
            txt_eke,
            transform=ax.transAxes,
            ha='right',
            va='bottom',
            fontsize=STYLE['annot_size'],
            color=STYLE['eke'],
            bbox=dict(facecolor='white', alpha=0.82, edgecolor='none', boxstyle='round,pad=0.2'),
        )

    ax.set_title(f'{region}')
    ax.set_xlabel('Year')
    if c == 0:
        ax.set_ylabel(r'Energy (cm$^2$ s$^{-2}$)')
    base_timeseries_style(ax)

axes[0].plot([], [], color=STYLE['ke'], lw=STYLE['smooth_lw'], label='KE')
axes[0].plot([], [], color=STYLE['eke'], lw=STYLE['smooth_lw'], label='EKE')
axes[0].legend(frameon=False, fontsize=8, loc='upper left')

fig.tight_layout()
out_fig = PLOT_DIR / 'figA2_ke_eke_timeseries_theilsen_mk.png'
fig.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_fig}')

# -----------------------------
# Figure A3: grouped bar trends
# -----------------------------
rows = []
for region in regions:
    grp = energy_ts[energy_ts['region'] == region].sort_values('decimal_year')
    x = grp['decimal_year'].values.astype(float)

    for var, label in [('mean_ke', 'KE'), ('mean_eke', 'EKE')]:
        y = grp[var].values.astype(float) * 1e4
        ts_res = theil_sen_with_ci(x, y)
        mk_res = mann_kendall_yue_wang(y)
        ci95 = 1.96 * ts_res['se'] if np.isfinite(ts_res['se']) else np.nan

        rows.append({
            'region': region,
            'variable': label,
            'slope': ts_res['slope'],
            'ci95': ci95,
            'mk_p': mk_res['p'],
        })

trend_df = pd.DataFrame(rows)
xpos = np.arange(len(regions))
w = 0.30

fig, ax = plt.subplots(figsize=(4.5, 4))

ke_df = trend_df[trend_df['variable'] == 'KE'].set_index('region').reindex(regions)
eke_df = trend_df[trend_df['variable'] == 'EKE'].set_index('region').reindex(regions)

ke_colors = [REGION_COLORS[r] if np.isfinite(ke_df.loc[r, 'mk_p']) and ke_df.loc[r, 'mk_p'] < 0.05 else '#C9CDD1' for r in regions]
eke_colors = [REGION_COLORS[r] if np.isfinite(eke_df.loc[r, 'mk_p']) and eke_df.loc[r, 'mk_p'] < 0.05 else '#D5DADF' for r in regions]

ax.bar(
    xpos - w / 2,
    ke_df['slope'].values,
    yerr=ke_df['ci95'].values,
    width=w,
    color=ke_colors,
    edgecolor='white',
    linewidth=0.6,
    error_kw={'lw': 1.0, 'capsize': 3.5, 'capthick': 1.0},
    zorder=3,
    label='KE',
)
ax.bar(
    xpos + w / 2,
    eke_df['slope'].values,
    yerr=eke_df['ci95'].values,
    width=w,
    color=eke_colors,
    edgecolor='white',
    linewidth=0.6,
    error_kw={'lw': 1.0, 'capsize': 3.5, 'capthick': 1.0},
    zorder=3,
    label='EKE',
)

# Significance stars
yr = np.nanmax(np.abs(np.r_[ke_df['slope'].values, eke_df['slope'].values]))
yr = max(yr, 1e-6)
for i, r in enumerate(regions):
    for xbar, sdf in [(xpos[i] - w / 2, ke_df), (xpos[i] + w / 2, eke_df)]:
        s = sdf.loc[r, 'slope']
        e = sdf.loc[r, 'ci95'] if np.isfinite(sdf.loc[r, 'ci95']) else 0.0
        p = sdf.loc[r, 'mk_p']
        st = stars(p)
        if st and np.isfinite(s):
            ytxt = s + e + 0.01 * yr if s >= 0 else s - e - 0.01 * yr
            ax.text(
                xbar,
                ytxt,
                st,
                ha='center',
                va='bottom' if s >= 0 else 'top',
                fontsize=10.5,
                fontweight='semibold',
                color='#1F2933',
            )

ax.axhline(0, color='#495057', lw=0.9)
ax.set_xticks(xpos)
ax.set_xticklabels(regions)
ax.set_ylabel(r'Trend (cm$^2$ s$^{-2}$ yr$^{-1}$)')
ax.grid(axis='y', alpha=STYLE['grid_alpha'], lw=STYLE['grid_lw'])
ax.legend(frameon=False, fontsize=9, loc='upper right')

fig.tight_layout()
out_fig = PLOT_DIR / 'figA3_ke_eke_bar_trends_theilsen_mk.png'
fig.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_fig}')

# Save trend summary table for reproducibility
trend_out = PLOT_DIR / 'tableA_ke_eke_theilsen_mk_trends.csv'
trend_df.to_csv(trend_out, index=False)
print(f'Saved: {trend_out}')

Saved: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/plots/trends_split/figA2_ke_eke_timeseries_theilsen_mk.png
Saved: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/plots/trends_split/figA3_ke_eke_bar_trends_theilsen_mk.png
Saved: /sessions/laughing-compassionate-darwin/mnt/OSR11/repository/outputs/plots/trends_split/tableA_ke_eke_theilsen_mk_trends.csv
